# Phase 5-B3-C｜已執行研究的「全新 process」重新評估（只讀，不消耗額度）

B3-C 的唯一 1 個 batch 已在 2026-11-10 14:17 執行完畢並結清（CONSUMED）。

執行當時，評估器和 Runner 跑在同一個 process 裡，因此 Phase 4-C 驗收的 **C8_governance_isolation 必然 FAIL**：C8 要求評估時不能載入 Runner 和 Core 模組。

這份 Notebook 只做一件事：在**沒有安裝 Core、也沒有載入 Runner 的全新 process** 裡，用既有的 Phase 4-C 評估器重新評估同一份 Runner 封存。

- **讀取來源**：Drive 上的 Runner 封存。MANIFEST 的 sha 必須等於執行紀錄裡的值。
- **治理狀態**：取自已推送的結果分支 `research/phase5b3c-result-p5b3c-0001`，其中含有這次執行的使用紀錄。
- **不做的事**：不重跑策略、不消耗額度、不讀 Holdout、不呼叫券商。
- **輸出**：
  - 重新評估的 8 個檔案；
  - 依新規則（B3C-08 必須 C1–C8 全部 PASS）算出的 B3C 驗收 v2。

  兩者都存到 Drive，並以**追加 commit** 推到同一個結果分支，不覆寫任何既有檔案。

**操作**：用全新 runtime，Run All。Colab Secrets 的 `GITHUB_TOKEN` 需要 Contents Read/Write 權限。


In [ ]:
# 第 1 格：所有人工核可集中在這裡；任何一項不成立立即停止
from pathlib import Path
import json, os, re, subprocess, sys, tempfile, urllib.request, hashlib, shutil
from datetime import datetime
from zoneinfo import ZoneInfo
REPO = "hh4832/strategy-research-agent"
CODE_BRANCH = "feature/phase5b3c-execution"
PIN = "ce7e36fb34f2b503a56cc639ba94f4ec63fa5cca"                                  # 重新評估程式碼；之後只允許 docs/、notebooks/ 變動
RESULT_BRANCH = "research/phase5b3c-result-p5b3c-0001"
RESULT_COMMIT = "444cfcd1f0180eea4ac76cef31e23696bd7de0ff"   # 執行後推送的紀錄（必須是結果分支的祖先）
ID = "RA-IBKR_TSM-SMA520_P5B3C-0001"
EVID_NAME = "20261110_141740_e3ce267"
APPROVER = "hh4832"
DRIVE_B3C = Path("/content/drive/MyDrive/00Quant_Research/phase5b3c") / ID
CODE, GOV = Path("/content/b3c_reeval_code"), Path("/content/b3c_reeval_gov")
TPE = ZoneInfo("Asia/Taipei")
assert sys.version_info >= (3, 11) and re.fullmatch(r"[0-9a-f]{40}", PIN)
assert not CODE.exists() and not GOV.exists(), "請 Disconnect and delete runtime 後重新 Run All（全新 runtime）"
from google.colab import drive, userdata
drive.mount("/content/drive")
assert (DRIVE_B3C / "CLAIM.json").is_file(), "Drive 上沒有 CLAIM.json：找不到已執行的那一次"
TOKEN = userdata.get("GITHUB_TOKEN")
assert TOKEN, "請在 Colab Secrets 設定 GITHUB_TOKEN"
req = urllib.request.Request("https://api.github.com/user", headers={"Authorization": "Bearer " + TOKEN,
                                                                    "Accept": "application/vnd.github+json"})
with urllib.request.urlopen(req, timeout=25) as r:
    login = json.load(r)["login"]
assert login.lower() == APPROVER, "GitHub token 所屬帳號不是研究負責人 hh4832"
runs = [p for p in sorted((DRIVE_B3C / "runs").iterdir()) if any((p / "archive/execution_records").glob("*.json"))]
assert len(runs) == 1, f"應恰好有一個含執行紀錄的 run：{runs}"
RUN = runs[0]
print("第 1 格 PASS：GitHub 身分", login, "｜Drive 已掛載｜全新 runtime｜run", RUN.name)
def run(*cmd, cwd=None, env=None, check=True):
    r = subprocess.run(cmd, cwd=cwd, env=env, text=True, capture_output=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{cmd[:3]} 失敗（exit {r.returncode}）：{r.stderr[-1500:]}")
    return r


In [ ]:
# 第 2 格：兩個乾淨 checkout（程式碼 PIN／已推送的治理紀錄）；不安裝 Core
run("git", "clone", "--quiet", f"https://github.com/{REPO}.git", str(CODE))
ref = "origin/main" if run("git", "merge-base", "--is-ancestor", PIN, "origin/main", cwd=CODE, check=False).returncode == 0 \
    else "origin/" + CODE_BRANCH
run("git", "checkout", "--quiet", "--detach", ref, cwd=CODE)
HEAD = run("git", "rev-parse", "HEAD", cwd=CODE).stdout.strip()
assert run("git", "merge-base", "--is-ancestor", PIN, HEAD, cwd=CODE, check=False).returncode == 0
drift = [f for f in run("git", "diff", "--name-only", PIN, HEAD, cwd=CODE).stdout.split() if f]
assert all(f.startswith(("docs/", "notebooks/")) for f in drift), f"Version Sync 失敗：{drift}"
run("git", "clone", "--quiet", "--branch", RESULT_BRANCH, f"https://github.com/{REPO}.git", str(GOV))
assert run("git", "merge-base", "--is-ancestor", RESULT_COMMIT, "HEAD", cwd=GOV, check=False).returncode == 0
run(sys.executable, "-m", "pip", "install", "--quiet", "-e", str(CODE))           # 不安裝 market-research-core
sys.path.insert(0, str(CODE / "src"))
from strategy_agent.execution import b3c_run as R
from strategy_agent.binding.store import BindingStore
assert R.ROOT == CODE
import importlib.util
assert importlib.util.find_spec("market_research_core") is None, "此 runtime 不應安裝 Core"
print("第 2 格 PASS：code", HEAD[:7], f"（{ref}）｜治理紀錄", run("git", "rev-parse", "HEAD", cwd=GOV).stdout.strip()[:7])


In [ ]:
# 第 3 格：全新 process 重新評估（只讀，0 batch）＋ B3C 驗收 v2
recs = sorted((RUN / "archive/execution_records").glob("*.json"))
assert len(recs) == 1
REC = recs[0]
pushed = GOV / "docs/evidence/phase5b3c" / EVID_NAME / "execution_record.json"
assert REC.read_bytes() == pushed.read_bytes(), "Drive 的執行紀錄與已推送的不同"
stamp = datetime.now(TPE).strftime("%Y%m%d_%H%M%S")
OUTD = DRIVE_B3C / "reevaluation" / stamp
RE = R.reevaluate(REC, OUTD / "evaluation", governance_root=GOV, evaluator_commit=HEAD,
                  archive_prefix_map=("/content/phase5b3c_run", str(RUN)))
print("重新評估：", RE["status"], RE["checks"], "｜evaluator process：", RE["evaluator_process"])
rec = json.loads(REC.read_text())
binding = BindingStore(GOV).for_bound(rec["contract"]["request_sha256"])
status = json.loads((GOV / "docs/evidence/phase5b3c" / EVID_NAME / "execution_status.json").read_text())
result = {**status, "execution_record": str(REC), "binding": binding}
ACC2 = R.acceptance(result, governance_root=GOV, cloud=True, production=True, evaluation=RE)
print("B3C v2：", {k: v["status"] for k, v in ACC2["gates"].items()}, "｜verdict", ACC2["verdict"])
DEST = GOV / "docs/evidence/phase5b3c" / f"{EVID_NAME}_fresh_reevaluation"
DEST.mkdir(parents=True, exist_ok=False)
shutil.copytree(OUTD / "evaluation", DEST / "evaluation")
for name, obj in (("reevaluation.json", RE), ("b3c_acceptance_v2.json", ACC2)):
    with open(DEST / name, "x") as f:
        f.write(json.dumps(obj, indent=2, sort_keys=True, ensure_ascii=False, default=str) + "\n")
idx = {str(p.relative_to(DEST)): hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(DEST.rglob("*")) if p.is_file()}
with open(DEST / "ARCHIVE_INDEX.json", "x") as f:
    f.write(json.dumps(idx, indent=2, sort_keys=True) + "\n")
shutil.copytree(DEST, OUTD / "evidence")
rd = DEST / "evaluation/research_decision.json"
if rd.is_file():
    print(json.dumps(json.loads(rd.read_text()), indent=1, ensure_ascii=False)[:1500])


In [ ]:
# 第 4 格：以追加 commit 推到結果分支（只新增 docs/evidence/phase5b3c/…_fresh_reevaluation/）
changed = sorted(f[3:] for f in run("git", "status", "--porcelain", "--untracked-files=all", cwd=GOV).stdout.splitlines())
assert changed and all(f.startswith(f"docs/evidence/phase5b3c/{EVID_NAME}_fresh_reevaluation/") for f in changed), changed
run("git", "add", "--", *changed, cwd=GOV)
run("git", "-c", f"user.name={APPROVER}", "-c", f"user.email={APPROVER}@users.noreply.github.com", "commit", "-q", "-m",
    f"research: Phase 5-B3-C fresh-process re-evaluation ({RE['status']}; B3C v2 {ACC2['verdict']}; 0 batches)", cwd=GOV)
with tempfile.TemporaryDirectory() as d:
    ask = Path(d) / "askpass.sh"
    ask.write_text("#!/bin/sh\ncase \"$1\" in *Username*) printf '%s' \"$GIT_USERNAME\";; *) printf '%s' \"$GIT_PASSWORD\";; esac\n")
    ask.chmod(0o700)
    env = dict(os.environ, GIT_ASKPASS=str(ask), GIT_TERMINAL_PROMPT="0", GIT_USERNAME="x-access-token", GIT_PASSWORD=TOKEN)
    res = subprocess.run(["git", "push", "origin", "HEAD:refs/heads/" + RESULT_BRANCH], cwd=GOV, env=env, text=True,
                         capture_output=True)
    env["GIT_PASSWORD"] = ""
if res.returncode != 0:
    raise RuntimeError(f"push 未完成（exit {res.returncode}）；結果已在 Drive：{OUTD}")
print("已推送", len(changed), "個檔案到", RESULT_BRANCH, "｜commit", run("git", "rev-parse", "HEAD", cwd=GOV).stdout.strip())
